In [1]:
# !pip install optuna lightgbm xgboost -q  # uncomment if needed
import re, time, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import optuna
import lightgbm as lgb
import xgboost as xgb
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore', category=UserWarning)

DATA_DIR = '/Users/hayden/coderepos_mac_mini/mitsui_commodity/data/'
HOLDOUT_START = 1827
VAL_FRAC      = 0.15

# Features (same as v2)
LAGS_POS     = [1, 2, 5]
LAGS_NEG     = [-5, -2, -1]
LAGS_AR      = [1, 5]
ROLL_WINDOWS = (5, 20)
INCLUDE_MACRO = True

# Transformer defaults
SEQ_LEN      = 20
BATCH_SIZE   = 64
EPOCHS       = 100
PATIENCE     = 15
WEIGHT_DECAY = 1e-4
SPEARMAN_TEMP = 0.5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── TIME BUDGET ──────────────────────────────────────────────────────────────
# Total tuning time = TUNE_HOURS. Adjust if your CPU is faster/slower.
# After tuning, full LGBM+XGB runs take ~30 min extra.
# Leave at 6 to finish comfortably within 7 hours.
TUNE_HOURS  = 6.0
LGBM_FRAC   = 0.25   # 1.5 h
XGB_FRAC    = 0.25   # 1.5 h
TF_FRAC     = 0.50   # 3.0 h

LGBM_SECONDS = int(TUNE_HOURS * 3600 * LGBM_FRAC)
XGB_SECONDS  = int(TUNE_HOURS * 3600 * XGB_FRAC)
TF_SECONDS   = int(TUNE_HOURS * 3600 * TF_FRAC)

train        = pd.read_csv(f'{DATA_DIR}/train.csv')
train_labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
target_pairs = pd.read_csv(f'{DATA_DIR}/target_pairs.csv')
train        = train.sort_values('date_id').reset_index(drop=True)
train_labels = train_labels.sort_values('date_id').reset_index(drop=True)
assert (train['date_id'].values == train_labels['date_id'].values).all()
target_cols = [c for c in train_labels.columns if c != 'date_id']
n_days = len(train)

print(f'train {train.shape}  labels {train_labels.shape}  device: {DEVICE}')
print(f'LGBM budget: {LGBM_SECONDS//60} min  XGB: {XGB_SECONDS//60} min  TF: {TF_SECONDS//60} min')
print(f'Total tuning: {TUNE_HOURS} h')

train (1961, 558)  labels (1961, 425)  device: cpu
LGBM budget: 90 min  XGB: 90 min  TF: 180 min
Total tuning: 6.0 h


In [2]:
def parse_pair(pair_str):
    s = pair_str.strip()
    tokens = re.split(r'\s*([+-])\s*', s)
    result = []
    if tokens[0]:
        result.append((1, tokens[0]))
    i = 1
    while i < len(tokens):
        op, ticker = tokens[i], tokens[i + 1]
        result.append((1 if op == '+' else -1, ticker))
        i += 2
    return result

def get_exchange(ticker):
    if ticker.startswith('US_Stock_'):
        return 'US_STOCK'
    return ticker.split('_')[0]

rows = []
for _, r in target_pairs.iterrows():
    tickers = [t for _, t in parse_pair(r['pair'])]
    exchanges = set(get_exchange(t) for t in tickers)
    rows.append({'target': r['target'], 'lag': r['lag'], 'pair_raw': r['pair'],
                 'tickers': tickers, 'exchange_key': '_'.join(sorted(exchanges))})
target_info = pd.DataFrame(rows)

def classify_ticker(t):
    fams = set()
    if 'JPX_Gold' in t:     fams.add('gold')
    if 'JPX_Platinum' in t: fams.add('platinum')
    if 'Rubber' in t:       fams.add('rubber')
    if 'LME_CA' in t:       fams.add('copper')
    if 'LME_AH' in t:       fams.add('aluminum')
    if 'LME_PB' in t:       fams.add('lead')
    if 'LME_ZS' in t:       fams.add('zinc')
    if any(s in t for s in ['_GLD_','_IAU_','_GDX_','_NEM_','_AEM_','_GOLD_','_KGC_','_FNV_','_WPM_','_SLV_']):
        fams.add('gold')
    if any(s in t for s in ['_FCX_','_SCCO_']):
        fams.add('copper')
    return fams

def families_for_tickers(tickers):
    fams = set()
    for t in tickers:
        fams |= classify_ticker(t)
    return fams

print(f'{len(target_info)} targets, {target_info["exchange_key"].nunique()} exchange groups')

424 targets, 8 exchange groups


In [3]:
def lret(s):
    return np.log(s / s.shift(1))

def build_engineered(train, lags=LAGS_POS):
    out = {}
    fam_cols = {f: [] for f in ['gold','platinum','rubber','copper','aluminum','lead','zinc']}
    series = {
        'usdjpy_ret':    lret(train['FX_USDJPY']),
        'gld_ret':       lret(train['US_Stock_GLD_adj_close']),
        'gsr':           train['US_Stock_GLD_adj_close'] / train['US_Stock_SLV_adj_close'].replace(0, np.nan),
        'plat_gold':     train['JPX_Platinum_Standard_Futures_Close'] / train['JPX_Gold_Standard_Futures_Close'].replace(0, np.nan),
        'xle_ret':       lret(train['US_Stock_XLE_adj_close']),
        'fxi_ret':       lret(train['US_Stock_FXI_adj_close']),
        'copper_gold':   train['LME_CA_Close'] / train['JPX_Gold_Standard_Futures_Close'].replace(0, np.nan),
        'fcx_ret':       lret(train['US_Stock_FCX_adj_close']),
        'xlb_ret':       lret(train['US_Stock_XLB_adj_close']),
    }
    series['gsr_ret']         = lret(series['gsr'])
    series['plat_gold_ret']   = lret(series['plat_gold'])
    series['copper_gold_ret'] = lret(series['copper_gold'])
    membership = {
        'usdjpy_ret':      ['gold','platinum','rubber'],
        'gld_ret':         ['gold'],
        'gsr':             ['gold'],
        'gsr_ret':         ['gold'],
        'plat_gold':       ['platinum'],
        'plat_gold_ret':   ['platinum'],
        'xle_ret':         ['rubber'],
        'fxi_ret':         ['rubber','copper'],
        'copper_gold':     ['copper','aluminum','lead','zinc'],
        'copper_gold_ret': ['copper','aluminum','lead','zinc'],
        'fcx_ret':         ['copper'],
        'xlb_ret':         ['aluminum','lead','zinc'],
    }
    for base, fams in membership.items():
        for k in lags:
            name = f'{base}_lag{k}'
            out[name] = series[base].shift(k)
            for f in fams:
                fam_cols[f].append(name)
    return pd.DataFrame(out).ffill(), fam_cols

eng_df, fam_cols = build_engineered(train)
print(f'eng_df: {eng_df.shape}')

eng_df: (1961, 36)


In [4]:
def own_target_features(target_name, train_labels, target_lag,
                        ar_lags=LAGS_AR, vol_windows=ROLL_WINDOWS):
    y = train_labels[target_name]
    parts = []
    for k in ar_lags:
        parts.append(y.shift(max(k, target_lag)).rename(f'own_lag{k}'))
    y_known = y.shift(target_lag)
    for w in vol_windows:
        parts.append(y_known.rolling(w).std().rename(f'own_vol{w}'))
        parts.append(y_known.rolling(w).mean().rename(f'own_mean{w}'))
    if len(vol_windows) >= 2:
        sv = y_known.rolling(vol_windows[0]).std()
        lv = y_known.rolling(vol_windows[-1]).std()
        parts.append((sv / (lv + 1e-8)).rename('own_vol_ratio'))
    return pd.concat(parts, axis=1)

def _resolve_price_col(ticker, cols):
    cset = set(cols)
    for cand in (ticker, f'{ticker}_Close', f'{ticker}_adj_close', f'{ticker}_close'):
        if cand in cset:
            return cand
    for c in cols:
        if c.startswith(ticker):
            return c
    return None

def ticker_features(target_row, train, pos_lags=LAGS_POS, neg_lags=LAGS_NEG, roll_windows=ROLL_WINDOWS):
    target_lag = target_row['lag']
    parts, missing = [], []
    for tk in target_row['tickers']:
        col = _resolve_price_col(tk, train.columns)
        if col is None:
            missing.append(tk)
            continue
        s = lret(train[col])
        for k in pos_lags:
            parts.append(s.shift(max(k, target_lag)).rename(f'{tk}_ret_lag{k}'))
        for k in neg_lags:
            parts.append(s.shift(k).rename(f'{tk}_ret_lag{k}'))
        for w in roll_windows:
            parts.append(s.rolling(w).mean().shift(target_lag).rename(f'{tk}_rmean{w}'))
    if not parts:
        return pd.DataFrame(index=train.index), missing
    return pd.concat(parts, axis=1).ffill(), missing

def features_for_target(target_row, eng_df, fam_cols, train_labels, train,
                        include_all_macro=False, include_macro=True):
    X_tk, missing = ticker_features(target_row, train)
    parts = [X_tk.reset_index(drop=True)]
    if include_macro or include_all_macro:
        if include_all_macro:
            X_cross = eng_df.copy()
        else:
            fams = families_for_tickers(target_row['tickers'])
            cols = sorted({c for f in fams for c in fam_cols.get(f, [])})
            X_cross = eng_df[cols] if cols else pd.DataFrame(index=eng_df.index)
        parts.append(X_cross.reset_index(drop=True))
    X_own = own_target_features(target_row['target'], train_labels, target_row['lag'])
    parts.append(X_own.reset_index(drop=True))
    return pd.concat(parts, axis=1), missing

In [6]:
def preprocess(X_train, *others):
    """Inf->NaN, median impute using train col medians, standardize."""
    def to_clean(X):
        X = np.array(X, dtype=np.float32)
        X[~np.isfinite(X)] = np.nan
        return X
    X_tr = to_clean(X_train)
    col_median = np.nanmedian(X_tr, axis=0)
    col_median = np.where(np.isnan(col_median), 0.0, col_median)
    def impute(X):
        X = to_clean(X).copy()
        for j in range(X.shape[1]):
            mask = np.isnan(X[:, j])
            if mask.any():
                X[mask, j] = col_median[j]
        return X
    X_tr_imp = impute(X_tr)
    mu  = X_tr_imp.mean(axis=0, keepdims=True)
    sig = X_tr_imp.std(axis=0, keepdims=True) + 1e-8
    return [(impute(arr) - mu) / sig for arr in (X_train, *others)]

In [7]:
target_X, target_y, target_cols_per_target, all_missing = {}, {}, {}, {}
for _, tr in target_info.iterrows():
    X, missing = features_for_target(tr, eng_df, fam_cols, train_labels, train,
                                     include_macro=INCLUDE_MACRO)
    target_X[tr['target']]                = X.values.astype(np.float32)
    target_y[tr['target']]                = train_labels[tr['target']].values.astype(np.float32)
    target_cols_per_target[tr['target']]  = X.columns.tolist()
    if missing:
        all_missing[tr['target']] = missing
n_feat_dist = pd.Series([v.shape[1] for v in target_X.values()]).value_counts().sort_index()
print(f'built {len(target_X)} targets  include_macro={INCLUDE_MACRO}')
print(f'feature count distribution:\n{n_feat_dist}')

built 424 targets  include_macro=True
feature count distribution:
15      4
32    253
35    128
38      9
41      6
44     18
47      6
Name: count, dtype: int64


In [8]:
def mitsui_metric(y_true, y_pred, verbose=False):
    daily_corrs = []
    for i in range(len(y_true)):
        mask = ~np.isnan(y_true[i])
        if mask.sum() < 2:
            continue
        corr, _ = spearmanr(y_true[i, mask], y_pred[i, mask])
        if not np.isnan(corr):
            daily_corrs.append(corr)
    arr = np.array(daily_corrs)
    std = arr.std()
    if std == 0:
        raise ZeroDivisionError('std of daily corrs is zero')
    sharpe = arr.mean() / std
    if verbose:
        print(f'mean {arr.mean():.4f}, std {std:.4f}, sharpe {sharpe:.4f} ndays {len(arr)}')
    return sharpe

train_mask   = train['date_id'] < HOLDOUT_START
holdout_mask = train['date_id'] >= HOLDOUT_START
n_train      = int(train_mask.sum())
split_idx    = int(n_train * (1 - VAL_FRAC))
n_holdout    = int(holdout_mask.sum())
y_hold_true  = train_labels.loc[holdout_mask, target_cols].values
print(f'n_train={n_train}, split_idx={split_idx}, holdout={n_holdout}')

n_train=1827, split_idx=1552, holdout=134


In [9]:
# ── Dataset ───────────────────────────────────────────────────────────────────
class SeqDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X, self.y, self.seq_len = torch.from_numpy(X), torch.from_numpy(y), seq_len
        self.valid = [i for i in range(len(X) - seq_len) if not np.isnan(y[i + seq_len])]
    def __len__(self):  return len(self.valid)
    def __getitem__(self, idx):
        i = self.valid[idx]
        return self.X[i:i + self.seq_len], self.y[i + self.seq_len]

# ── Soft Spearman loss ────────────────────────────────────────────────────────
def soft_rank(x, temp=SPEARMAN_TEMP):
    return torch.sigmoid((x.unsqueeze(0) - x.unsqueeze(1)) / temp).sum(dim=1)

def soft_spearman_loss(pred, target, temp=SPEARMAN_TEMP):
    rp = soft_rank(pred, temp)
    with torch.no_grad():
        rt = soft_rank(target, temp)
    rp, rt = rp - rp.mean(), rt - rt.mean()
    return 1.0 - (rp * rt).sum() / (rp.norm() * rt.norm() + 1e-8)

# ── Transformer ───────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerRegressor(nn.Module):
    def __init__(self, n_feat, d_model=64, n_heads=4, n_layers=2, dropout=0.1, ff_mult=4):
        super().__init__()
        assert d_model % n_heads == 0, f'd_model={d_model} not divisible by n_heads={n_heads}'
        self.proj    = nn.Linear(n_feat, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        enc_layer    = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * ff_mult,
            dropout=dropout, batch_first=True, norm_first=True)
        self.encoder  = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.dropout  = nn.Dropout(dropout)
        self.head     = nn.Linear(d_model, 1)
    def forward(self, x):
        x = self.pos_enc(self.proj(x))
        x = self.encoder(x)
        return self.head(self.dropout(x[:, -1, :])).squeeze(-1)

print('Models defined: TransformerRegressor')

Models defined: TransformerRegressor


In [10]:
def train_transformer_target(target_name, hp, seed=0):
    """hp keys: d_model, n_heads, n_layers, dropout, lr, seq_len, batch_size, patience, max_epochs"""
    torch.manual_seed(seed)
    d_model    = hp['d_model']
    n_heads    = hp['n_heads']
    n_layers   = hp['n_layers']
    dropout    = hp['dropout']
    lr         = hp['lr']
    seq_len    = hp.get('seq_len',    SEQ_LEN)
    batch_size = hp.get('batch_size', BATCH_SIZE)
    patience   = hp.get('patience',   PATIENCE)
    max_epochs = hp.get('max_epochs', EPOCHS)

    X_full = target_X[target_name]
    y_full = target_y[target_name]
    X_tr, X_val, X_ho = preprocess(X_full[:split_idx], X_full[split_idx:n_train], X_full[n_train:])
    y_tr, y_val, y_ho = y_full[:split_idx], y_full[split_idx:n_train], y_full[n_train:]

    train_ds = SeqDataset(X_tr,  y_tr,  seq_len)
    val_ds   = SeqDataset(X_val, y_val, seq_len)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    model = TransformerRegressor(X_full.shape[1], d_model, n_heads, n_layers, dropout).to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)

    best_sp, best_state, p_ctr = -np.inf, None, 0
    for epoch in range(max_epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            soft_spearman_loss(model(xb), yb).backward()
            opt.step()
        model.eval()
        vp, vt = [], []
        with torch.no_grad():
            for xb, yb in val_dl:
                vp.extend(model(xb.to(DEVICE)).cpu().numpy())
                vt.extend(yb.numpy())
        val_sp = spearmanr(vt, vp)[0] if len(vp) >= 2 else -1.0
        if np.isnan(val_sp): val_sp = -1.0
        if val_sp > best_sp:
            best_sp    = val_sp
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            p_ctr      = 0
        else:
            p_ctr += 1
            if p_ctr >= patience:
                break
    if best_state:
        model.load_state_dict(best_state)
    X_hist = np.concatenate([X_tr, X_val, X_ho])
    preds  = np.zeros(len(y_ho), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for j in range(len(y_ho)):
            end = n_train + j
            w   = X_hist[end - seq_len:end]
            preds[j] = model(torch.from_numpy(w).unsqueeze(0).to(DEVICE)).item()
    return preds


def train_lgbm_target(target_name, params):
    X_full = target_X[target_name]
    y_full = target_y[target_name]
    X_tr, X_val, X_ho = preprocess(X_full[:split_idx], X_full[split_idx:n_train], X_full[n_train:])
    y_tr, y_val, y_ho = y_full[:split_idx], y_full[split_idx:n_train], y_full[n_train:]
    ok_tr, ok_val = ~np.isnan(y_tr), ~np.isnan(y_val)
    mdl = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)
    mdl.fit(X_tr[ok_tr], y_tr[ok_tr],
            eval_set=[(X_val[ok_val], y_val[ok_val])],
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    return mdl.predict(X_ho).astype(np.float32)


def train_xgb_target(target_name, params):
    X_full = target_X[target_name]
    y_full = target_y[target_name]
    X_tr, X_val, X_ho = preprocess(X_full[:split_idx], X_full[split_idx:n_train], X_full[n_train:])
    y_tr, y_val, y_ho = y_full[:split_idx], y_full[split_idx:n_train], y_full[n_train:]
    ok_tr, ok_val = ~np.isnan(y_tr), ~np.isnan(y_val)
    mdl = xgb.XGBRegressor(**params, random_state=42, verbosity=0)
    mdl.fit(X_tr[ok_tr], y_tr[ok_tr],
            eval_set=[(X_val[ok_val], y_val[ok_val])],
            early_stopping_rounds=50, verbose=False)
    return mdl.predict(X_ho).astype(np.float32)

print('Training functions defined: train_transformer_target, train_lgbm_target, train_xgb_target')

Training functions defined: train_transformer_target, train_lgbm_target, train_xgb_target


In [11]:
# Tuning subset: 2-3 targets per exchange group (~20 total)
TUNE_SUBSET = []
for ekey in sorted(target_info['exchange_key'].unique()):
    grp = target_info[target_info['exchange_key'] == ekey]['target'].tolist()
    n   = max(2, min(3, len(grp) // 15))
    TUNE_SUBSET.extend(grp[:n])
print(f'Tune subset: {len(TUNE_SUBSET)} targets')
print(TUNE_SUBSET)

TUNE_Y_IDX  = [target_cols.index(t) for t in TUNE_SUBSET]
TUNE_Y_TRUE = y_hold_true[:, TUNE_Y_IDX]

def tune_sharpe(pred_fn):
    """Train pred_fn on each target in TUNE_SUBSET, return holdout Sharpe."""
    n     = len(TUNE_SUBSET)
    preds = np.full((n_holdout, n), np.nan, dtype=np.float32)
    for i, t in enumerate(TUNE_SUBSET):
        preds[:, i] = pred_fn(t)
    try:
        return mitsui_metric(TUNE_Y_TRUE, preds)
    except ZeroDivisionError:
        return 0.0

Tune subset: 20 targets
['target_212', 'target_318', 'target_12', 'target_14', 'target_19', 'target_9', 'target_11', 'target_13', 'target_4', 'target_5', 'target_8', 'target_10', 'target_17', 'target_2', 'target_3', 'target_1', 'target_7', 'target_16', 'target_0', 'target_106']


In [12]:
# ── Optuna objectives ─────────────────────────────────────────────────────────

def lgbm_objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 100, 2000),
        'num_leaves':        trial.suggest_int('num_leaves', 15, 255),
        'max_depth':         trial.suggest_int('max_depth', -1, 15),
        'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
    }
    return tune_sharpe(lambda t: train_lgbm_target(t, params))


def xgb_objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'learning_rate':    trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'tree_method':      'hist',
    }
    return tune_sharpe(lambda t: train_xgb_target(t, params))


def transformer_objective(trial):
    d_model  = trial.suggest_categorical('d_model', [32, 64, 128])
    # n_heads must divide d_model
    ok_heads = [h for h in [2, 4, 8] if d_model % h == 0]
    hp = {
        'd_model':    d_model,
        'n_heads':    trial.suggest_categorical('n_heads', ok_heads),
        'n_layers':   trial.suggest_int('n_layers', 1, 4),
        'dropout':    trial.suggest_float('dropout', 0.05, 0.5),
        'lr':         trial.suggest_float('lr', 5e-5, 5e-3, log=True),
        'seq_len':    trial.suggest_categorical('seq_len', [10, 20, 30, 40]),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'patience':   trial.suggest_int('patience', 5, 20),
        'max_epochs': 100,
    }
    return tune_sharpe(lambda t: train_transformer_target(t, hp))

print('Optuna objectives defined')

Optuna objectives defined


In [13]:
# ── Default / fallback hyperparameters ───────────────────────────────────────
# These are used if you skip tuning and go straight to full training.
# The overnight tuning cells will overwrite these with better values.
BEST_LGBM_PARAMS = {
    'n_estimators': 500, 'num_leaves': 63, 'max_depth': -1,
    'learning_rate': 0.05, 'min_child_samples': 20,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
}
BEST_XGB_PARAMS = {
    'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 1.0, 'min_child_weight': 3,
    'tree_method': 'hist',
}
BEST_TF_PARAMS = {
    'd_model': 64, 'n_heads': 4, 'n_layers': 2, 'dropout': 0.2,
    'lr': 1e-3, 'seq_len': SEQ_LEN, 'batch_size': 64,
    'patience': 15, 'max_epochs': 100,
}
print('Fallback hyperparameters set.')

Fallback hyperparameters set.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SMOKE TEST — run this before starting the overnight session.
# Expected runtime: 3-8 minutes. Fix any errors before proceeding.
# ═══════════════════════════════════════════════════════════════════════════════
_smoke_targets = TUNE_SUBSET[:2]
_smoke_y_idx   = [target_cols.index(t) for t in _smoke_targets]

def _smoke_sharpe(pred_fn):
    preds = np.full((n_holdout, 2), np.nan, dtype=np.float32)
    for i, t in enumerate(_smoke_targets):
        preds[:, i] = pred_fn(t)
    try:
        return mitsui_metric(y_hold_true[:, _smoke_y_idx], preds)
    except ZeroDivisionError:
        return 0.0

errors = []
t0 = time.time()

# --- LGBM training function ---
try:
    _p = _smoke_sharpe(lambda t: train_lgbm_target(t, BEST_LGBM_PARAMS))
    print(f'  [OK] LGBM train       sharpe={_p:.4f}')
except Exception as e:
    errors.append(f'LGBM train: {e}')
    print(f'  [FAIL] LGBM train: {e}')

# --- XGB training function ---
try:
    _p = _smoke_sharpe(lambda t: train_xgb_target(t, BEST_XGB_PARAMS))
    print(f'  [OK] XGB  train       sharpe={_p:.4f}')
except Exception as e:
    errors.append(f'XGB train: {e}')
    print(f'  [FAIL] XGB train: {e}')

# --- Transformer training function (3 epochs) ---
try:
    _tf_smoke = {**BEST_TF_PARAMS, 'max_epochs': 3, 'patience': 2}
    _p = _smoke_sharpe(lambda t: train_transformer_target(t, _tf_smoke))
    print(f'  [OK] TF   train       sharpe={_p:.4f}')
except Exception as e:
    errors.append(f'TF train: {e}')
    print(f'  [FAIL] TF train: {e}')

# --- LGBM Optuna (1 trial) ---
try:
    _s = optuna.create_study(direction='maximize')
    _s.optimize(lgbm_objective, n_trials=1)
    print(f'  [OK] LGBM Optuna      trial_sharpe={_s.best_value:.4f}')
except Exception as e:
    errors.append(f'LGBM Optuna: {e}')
    print(f'  [FAIL] LGBM Optuna: {e}')

# --- XGB Optuna (1 trial) ---
try:
    _s = optuna.create_study(direction='maximize')
    _s.optimize(xgb_objective, n_trials=1)
    print(f'  [OK] XGB  Optuna      trial_sharpe={_s.best_value:.4f}')
except Exception as e:
    errors.append(f'XGB Optuna: {e}')
    print(f'  [FAIL] XGB Optuna: {e}')

# --- Transformer Optuna (1 trial, capped at 3 epochs via monkey-patch) ---
try:
    def _tf_smoke_obj(trial):
        d_model  = trial.suggest_categorical('d_model', [32, 64])
        ok_heads = [h for h in [2, 4] if d_model % h == 0]
        hp = {
            'd_model':    d_model,
            'n_heads':    trial.suggest_categorical('n_heads', ok_heads),
            'n_layers':   1,
            'dropout':    trial.suggest_float('dropout', 0.05, 0.3),
            'lr':         trial.suggest_float('lr', 5e-4, 2e-3, log=True),
            'seq_len':    SEQ_LEN,
            'batch_size': 64,
            'patience':   2,
            'max_epochs': 3,
        }
        return _smoke_sharpe(lambda t: train_transformer_target(t, hp))
    _s = optuna.create_study(direction='maximize')
    _s.optimize(_tf_smoke_obj, n_trials=1)
    print(f'  [OK] TF   Optuna      trial_sharpe={_s.best_value:.4f}')
except Exception as e:
    errors.append(f'TF Optuna: {e}')
    print(f'  [FAIL] TF Optuna: {e}')

print(f'\nSmoke test took {(time.time()-t0)/60:.1f} min')
if errors:
    print(f'\n!!! SMOKE TEST FAILED ({len(errors)} errors) — fix before overnight run !!!')
    for e in errors: print(f'  {e}')
else:
    total_h = (LGBM_SECONDS + XGB_SECONDS + TF_SECONDS) / 3600
    print(f'\n=== SMOKE TEST PASSED ===')
    print(f'Start overnight run. Estimated tuning time: {total_h:.1f} h')
    print(f'Set TUNE_HOURS lower if you need a shorter run.')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OVERNIGHT — LGBM TUNING
# ═══════════════════════════════════════════════════════════════════════════════
print(f'Starting LGBM tuning ({LGBM_SECONDS//60} min)...')
lgbm_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
lgbm_study.optimize(lgbm_objective, timeout=LGBM_SECONDS, show_progress_bar=True)
BEST_LGBM_PARAMS = lgbm_study.best_params
print(f'\nLGBM done. best_sharpe={lgbm_study.best_value:.4f}  n_trials={len(lgbm_study.trials)}')
print(f'Best params: {BEST_LGBM_PARAMS}')

Starting LGBM tuning (90 min)...


   0%|          | 00:00/1:30:00

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OVERNIGHT — XGB TUNING
# ═══════════════════════════════════════════════════════════════════════════════
print(f'Starting XGB tuning ({XGB_SECONDS//60} min)...')
xgb_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
xgb_study.optimize(xgb_objective, timeout=XGB_SECONDS, show_progress_bar=True)
BEST_XGB_PARAMS = {**xgb_study.best_params, 'tree_method': 'hist'}
print(f'\nXGB done. best_sharpe={xgb_study.best_value:.4f}  n_trials={len(xgb_study.trials)}')
print(f'Best params: {BEST_XGB_PARAMS}')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# OVERNIGHT — TRANSFORMER TUNING
# ═══════════════════════════════════════════════════════════════════════════════
print(f'Starting Transformer tuning ({TF_SECONDS//60} min)...')
tf_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
tf_study.optimize(transformer_objective, timeout=TF_SECONDS, show_progress_bar=True)
bp = tf_study.best_params
BEST_TF_PARAMS = {
    'd_model':    bp['d_model'],
    'n_heads':    bp['n_heads'],
    'n_layers':   bp['n_layers'],
    'dropout':    bp['dropout'],
    'lr':         bp['lr'],
    'seq_len':    bp.get('seq_len', SEQ_LEN),
    'batch_size': bp.get('batch_size', BATCH_SIZE),
    'patience':   bp.get('patience', PATIENCE),
    'max_epochs': 100,
}
print(f'\nTF done. best_sharpe={tf_study.best_value:.4f}  n_trials={len(tf_study.trials)}')
print(f'Best params: {BEST_TF_PARAMS}')

In [ ]:
# ── Tuning summary ────────────────────────────────────────────────────────────
print('=== TUNING RESULTS ===')
print(f'  LGBM  subset_sharpe={lgbm_study.best_value:.4f}  trials={len(lgbm_study.trials)}')
print(f'  XGB   subset_sharpe={xgb_study.best_value:.4f}  trials={len(xgb_study.trials)}')
print(f'  TF    subset_sharpe={tf_study.best_value:.4f}  trials={len(tf_study.trials)}')
print()
print('BEST_LGBM_PARAMS =', BEST_LGBM_PARAMS)
print('BEST_XGB_PARAMS  =', BEST_XGB_PARAMS)
print('BEST_TF_PARAMS   =', BEST_TF_PARAMS)

In [ ]:
# ── Full LGBM run — all 424 targets ──────────────────────────────────────────
lgbm_preds = np.full_like(y_hold_true, np.nan, dtype=np.float32)
t0 = time.time()
for ti, t in enumerate(target_cols):
    lgbm_preds[:, ti] = train_lgbm_target(t, BEST_LGBM_PARAMS)
    if (ti + 1) % 50 == 0:
        elapsed = (time.time() - t0) / 60
        print(f'  LGBM {ti+1}/{len(target_cols)}  elapsed={elapsed:.1f} min')
lgbm_score = mitsui_metric(y_hold_true, lgbm_preds, verbose=True)
print(f'LGBM holdout Sharpe: {lgbm_score:.4f}')

In [ ]:
# ── Full XGB run — all 424 targets ───────────────────────────────────────────
xgb_preds = np.full_like(y_hold_true, np.nan, dtype=np.float32)
t0 = time.time()
for ti, t in enumerate(target_cols):
    xgb_preds[:, ti] = train_xgb_target(t, BEST_XGB_PARAMS)
    if (ti + 1) % 50 == 0:
        elapsed = (time.time() - t0) / 60
        print(f'  XGB  {ti+1}/{len(target_cols)}  elapsed={elapsed:.1f} min')
xgb_score = mitsui_metric(y_hold_true, xgb_preds, verbose=True)
print(f'XGB holdout Sharpe: {xgb_score:.4f}')

In [ ]:
# ── Full Transformer run — all 424 targets ────────────────────────────────────
# NOTE: on CPU this takes ~1-3 h depending on best seq_len / epochs found.
# Run this after reviewing the tuning results if time permits.
tf_preds = np.full_like(y_hold_true, np.nan, dtype=np.float32)
t0 = time.time()
for ti, t in enumerate(target_cols):
    tf_preds[:, ti] = train_transformer_target(t, BEST_TF_PARAMS)
    if (ti + 1) % 25 == 0:
        elapsed = (time.time() - t0) / 60
        print(f'  TF  {ti+1}/{len(target_cols)}  elapsed={elapsed:.1f} min')
tf_score = mitsui_metric(y_hold_true, tf_preds, verbose=True)
print(f'Transformer holdout Sharpe: {tf_score:.4f}')

In [ ]:
# ── Model comparison ──────────────────────────────────────────────────────────
results = {'LGBM': lgbm_score, 'XGB': xgb_score, 'Transformer': tf_score}
print('=== Holdout Sharpe Comparison ===')
for name, score in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {name:<15} {score:+.4f}')

In [ ]:
# ── Per-group breakdown for best model ───────────────────────────────────────
best_name  = max(results, key=results.get)
best_preds = {'LGBM': lgbm_preds, 'XGB': xgb_preds, 'Transformer': tf_preds}[best_name]
print(f'=== Per-group Sharpe ({best_name}) ===')
for ekey in sorted(target_info['exchange_key'].unique()):
    grp    = target_info[target_info['exchange_key'] == ekey]['target'].tolist()
    y_idx  = [target_cols.index(t) for t in grp]
    g_true = y_hold_true[:, y_idx]
    g_pred = best_preds[:, y_idx]
    try:
        s = mitsui_metric(g_true, g_pred)
        print(f'  {ekey:<20} {s:+.4f}  ({len(grp)} targets)')
    except ZeroDivisionError as e:
        print(f'  {ekey:<20} ERROR: {e}')